In [11]:
"""
NB00_Project_Setup_and_Constants

Purpose
-------
1) Define global constants and default run parameters.
2) Provide lightweight logging utilities for reproducibility.
3) Confirm environment basics (versions, CPU info).
4) Define a canonical default orbit for downstream notebooks.
5) Provide a spatial-resolution sanity check utility.

Outputs (in-memory)
-------------------
- CONFIG          : dict of constants + default parameters
- helper functions : make_run_id(), save_manifest(), check_spatial_resolution()
- NB00_EXPORTS    : list of names downstream notebooks should rely on

Note
----
We are NOT creating a folder structure yet; all outputs can be saved to the current directory.
"""

'\nNB00_Project_Setup_and_Constants\n\nPurpose\n-------\n1) Define global constants and default run parameters.\n2) Provide lightweight logging utilities for reproducibility.\n3) Confirm environment basics (versions, CPU info).\n4) Define a canonical default orbit for downstream notebooks.\n5) Provide a spatial-resolution sanity check utility.\n\nOutputs (in-memory)\n-------------------\n- CONFIG          : dict of constants + default parameters\n- helper functions : make_run_id(), save_manifest(), check_spatial_resolution()\n- NB00_EXPORTS    : list of names downstream notebooks should rely on\n\nNote\n----\nWe are NOT creating a folder structure yet; all outputs can be saved to the current directory.\n'

In [12]:
# ============================================================
# Cell 1 : Imports and Environment Check
# ============================================================
import os
import sys
import json
import time
import platform
import hashlib
from dataclasses import dataclass, asdict
from datetime import datetime, timezone

import numpy as np

# Optional but usually available; we won't hard-require these yet.
try:
    import scipy
    from scipy import __version__ as scipy_version
except Exception:
    scipy = None
    scipy_version = None

try:
    import matplotlib
    import matplotlib.pyplot as plt
    from matplotlib import __version__ as mpl_version
except Exception:
    matplotlib = None
    plt = None
    mpl_version = None

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Processor:", platform.processor())
print("NumPy:", np.__version__)
print("SciPy:", scipy_version)
print("Matplotlib:", mpl_version)
print("CWD:", os.getcwd())

Python: 3.12.4 (tags/v3.12.4:8e8a4ba, Jun  6 2024, 19:30:16) [MSC v.1940 64 bit (AMD64)]
Platform: Windows-11-10.0.26200-SP0
Processor: AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD
NumPy: 2.3.5
SciPy: 1.17.0
Matplotlib: 3.10.8
CWD: c:\Users\hp\Desktop\RYZEN\PYTHON PROGRAMMING\OSADUGBA'S EQUATION


In [13]:
# ============================================================
# Cell 2 : NumPy Print Settings and Machine Epsilon
# ============================================================
np.set_printoptions(precision=6, suppress=True, linewidth=140)

EPS = np.finfo(float).eps
print("Machine epsilon:", EPS)

Machine epsilon: 2.220446049250313e-16


In [14]:
# ============================================================
# Cell 3 : Lunar Physical Constants
# ============================================================
# Units throughout: km, s, rad
#
# GM  : GRAIL-derived value from JPL DE430 / GL0900D gravity solution.
#        Source: Konopliv et al. (2014), "The JPL lunar gravity field to
#        spherical harmonic degree 900", Table 2.
#        Update this if your coefficient file header carries a different GM.
#
# R   : Mean lunar reference radius used in the GRAIL spherical-harmonic
#        expansion (consistent with GL0900D).
#
# omega: Sidereal rotation rate.  Period = 27.321661 days = 2,360,591.4 s
#         => omega = 2*pi / 2360591.4 ≈ 2.6616995e-6 rad/s.

MU_MOON_KM3_S2  = 4902.800076       # [km^3/s^2]  GRAIL GL0900D
R_MOON_KM       = 1737.4            # [km]         mean reference radius
OMEGA_MOON_RAD_S = 2.6616995e-6     # [rad/s]      sidereal: 2*pi / 2360591.4 s

print(f"MU_MOON_KM3_S2 : {MU_MOON_KM3_S2}")
print(f"R_MOON_KM      : {R_MOON_KM}")
print(f"OMEGA_MOON_RAD_S: {OMEGA_MOON_RAD_S:.7e}")

MU_MOON_KM3_S2 : 4902.800076
R_MOON_KM      : 1737.4
OMEGA_MOON_RAD_S: 2.6616995e-06


In [15]:
# ============================================================
# Cell 4 : CONFIG Dictionary
# ============================================================
CONFIG = {
    # --- Physical constants ---
    "mu_km3_s2":        MU_MOON_KM3_S2,
    "R_moon_km":        R_MOON_KM,
    "omega_moon_rad_s": OMEGA_MOON_RAD_S,

    # --- Gravity model settings ---
    "L_default": 200,       # default truncation degree/order

    # --- Nominal orbit discretization ---
    "M_default": 4000,      # anomaly samples per revolution

    # --- Default canonical test orbit (Section 3.1) ---
    # 50 km circular polar orbit — consistent with typical LLO ops.
    # Downstream notebooks fall back to this if no orbit is specified.
    "default_orbit": {
        "a_km":      R_MOON_KM + 50.0,   # 1787.4 km
        "e":         0.0,
        "i_deg":     90.0,                # polar
        "RAAN_deg":  0.0,
        "omega_deg": 0.0,
        "nu0_deg":   0.0,
    },

    # --- Encounter gating / segmentation (Section 3.3, 5.4) ---
    "gate": {
        "mode":       "percentile",   # "absolute" or "percentile"
        "percentile": 90.0,           # used when mode == "percentile"
        "g_th_abs":   None,           # used when mode == "absolute" [km/s^2]
        "smoothing": {
            "enabled":       True,
            "method":        "moving_average",
            "window_points": 31        # odd integer recommended
        },
        # Hysteresis (Section 3.3): encounter BEGINS when ||dg|| >= g_high
        # and TERMINATES only when ||dg|| <= g_low, with g_high > g_low.
        # high_multiplier and low_multiplier are applied to the base
        # threshold g_th to produce g_high and g_low respectively.
        "hysteresis": {
            "enabled":         False,
            "high_multiplier": 1.05,   # g_high = 1.05 * g_th  (entry)
            "low_multiplier":  0.95    # g_low  = 0.95 * g_th  (exit)
        },
        "min_window_points": 20,       # discard encounters shorter than this
        "merge_gap_points":  10        # merge windows separated by <= this gap
    },

    # --- Quadrature ---
    "quadrature": "trapezoid",          # also supports "simpson" in later NB

    # --- Truth propagation (validation notebooks) ---
    "truth_integrator": {
        "method": "DOP853",
        "rtol":   1e-11,
        "atol":   1e-12
    }
}

print(json.dumps(CONFIG, indent=2))

{
  "mu_km3_s2": 4902.800076,
  "R_moon_km": 1737.4,
  "omega_moon_rad_s": 2.6616995e-06,
  "L_default": 200,
  "M_default": 4000,
  "default_orbit": {
    "a_km": 1787.4,
    "e": 0.0,
    "i_deg": 90.0,
    "RAAN_deg": 0.0,
    "omega_deg": 0.0,
    "nu0_deg": 0.0
  },
  "gate": {
    "mode": "percentile",
    "percentile": 90.0,
    "g_th_abs": null,
    "smoothing": {
      "enabled": true,
      "method": "moving_average",
      "window_points": 31
    },
    "hysteresis": {
      "enabled": false,
      "high_multiplier": 1.05,
      "low_multiplier": 0.95
    },
    "min_window_points": 20,
    "merge_gap_points": 10
  },
  "quadrature": "trapezoid",
  "truth_integrator": {
    "method": "DOP853",
    "rtol": 1e-11,
    "atol": 1e-12
  }
}


In [16]:
# ============================================================
# Cell 5 : Spatial Resolution Sanity Check
# ============================================================
def check_spatial_resolution(a_km: float, L: int, M: int,
                              R_km: float = R_MOON_KM) -> dict:
    """
    Verify that the anomaly grid M is dense enough to resolve
    gravity features at truncation degree L for a circular orbit
    of semimajor axis a_km.

    Returns a dict with diagnostic values.

    Rule of thumb (Section 5.3): along-track step << pi * R / L
    """
    circumference_km   = 2.0 * np.pi * a_km
    step_km            = circumference_km / M
    half_wavelength_km = np.pi * R_km / L      # surface half-wavelength
    samples_per_half   = half_wavelength_km / step_km

    ok = samples_per_half >= 4.0   # conservative: at least 4 samples per feature

    result = {
        "a_km":                a_km,
        "L":                   L,
        "M":                   M,
        "orbit_circumference_km": round(circumference_km, 2),
        "along_track_step_km":    round(step_km, 3),
        "half_wavelength_km":     round(half_wavelength_km, 3),
        "samples_per_half_wave":  round(samples_per_half, 1),
        "resolution_adequate":    ok,
    }
    return result


# Run check for the default orbit and default L, M
_orb = CONFIG["default_orbit"]
_chk = check_spatial_resolution(
    a_km=_orb["a_km"],
    L=CONFIG["L_default"],
    M=CONFIG["M_default"]
)
print("Spatial resolution check:")
for k, v in _chk.items():
    print(f"  {k:30s} : {v}")

Spatial resolution check:
  a_km                           : 1787.4
  L                              : 200
  M                              : 4000
  orbit_circumference_km         : 11230.57
  along_track_step_km            : 2.808
  half_wavelength_km             : 27.291
  samples_per_half_wave          : 9.7
  resolution_adequate            : True


In [17]:
# ============================================================
# Cell 6 : Utility Functions and Manifest
# ============================================================
class _NumpyEncoder(json.JSONEncoder):
    """JSON encoder that handles NumPy scalars and arrays."""
    def default(self, obj):
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)


def sha256_of_dict(d: dict) -> str:
    """Deterministic SHA-256 hash of a JSON-serialisable dict."""
    b = json.dumps(d, sort_keys=True, indent=None, cls=_NumpyEncoder).encode("utf-8")
    return hashlib.sha256(b).hexdigest()


def make_run_id(prefix: str = "run") -> str:
    """Timestamp-based run ID (UTC)."""
    ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    return f"{prefix}_{ts}"


def save_manifest(path: str, manifest: dict) -> None:
    """Save a run manifest to JSON with a UTC timestamp."""
    manifest = dict(manifest)  # shallow copy
    manifest["saved_at_utc"] = datetime.now(timezone.utc).isoformat()
    with open(path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, sort_keys=True, cls=_NumpyEncoder)


# --- Build manifest for this notebook run ---
RUN_ID = make_run_id("NB00")
MANIFEST = {
    "run_id":      RUN_ID,
    "notebook":    "NB00_Project_Setup_and_Constants",
    "config_hash": sha256_of_dict(CONFIG),
    "config":      CONFIG,
    "env": {
        "python":     sys.version,
        "platform":   platform.platform(),
        "numpy":      np.__version__,
        "scipy":      scipy_version,
        "matplotlib": mpl_version,
        "cwd":        os.getcwd(),
    }
}

print("RUN_ID:", RUN_ID)
print("CONFIG hash:", MANIFEST["config_hash"])

RUN_ID: NB00_20260316T033515Z
CONFIG hash: 45d33af838c780a98a0a7a631f636467305466839525ce3a994fb64557de7678


In [18]:
# ============================================================
# Cell 7 : Save Manifest
# ============================================================
manifest_filename = f"manifest_{RUN_ID}.json"
save_manifest(manifest_filename, MANIFEST)
print("Saved:", manifest_filename)

Saved: manifest_NB00_20260316T033515Z.json


In [19]:
# ============================================================
# Cell 8 : Exports Summary
# ============================================================
# Downstream notebooks that %run or import from NB00 should
# rely on the following names.

NB00_EXPORTS = [
    # Constants
    "MU_MOON_KM3_S2",
    "R_MOON_KM",
    "OMEGA_MOON_RAD_S",
    "EPS",
    # Config
    "CONFIG",
    # Utility functions
    "make_run_id",
    "save_manifest",
    "sha256_of_dict",
    "check_spatial_resolution",
    # JSON helper
    "_NumpyEncoder",
]

print("NB00 exports the following names:")
for name in NB00_EXPORTS:
    print(f"  {name}")

print("\n--- NB00 setup complete ---")

NB00 exports the following names:
  MU_MOON_KM3_S2
  R_MOON_KM
  OMEGA_MOON_RAD_S
  EPS
  CONFIG
  make_run_id
  save_manifest
  sha256_of_dict
  check_spatial_resolution
  _NumpyEncoder

--- NB00 setup complete ---
